In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score



# 1. Split data and perform feature scaling


In [6]:
def split_scalar(indep_X, dep_Y):
    X_train, X_test, y_train, y_test = train_test_split(indep_X,dep_Y,test_size=0.25,random_state=0)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)
    return X_train, X_test, y_train, y_test

# 2. Calculate R2 score

In [7]:
def r2_prediction(regressor, X_test, y_test):
    y_pred = regressor.predict(X_test)
    return r2_score(y_test, y_pred)

# 3.Different Algorithms

In [8]:
# --------------------------------------------------
# 3. Linear Regression
# --------------------------------------------------
def Linear(X_train, y_train, X_test, y_test):
    regressor = LinearRegression()
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)
# --------------------------------------------------
# 4. Linear SVR
# --------------------------------------------------
def svm_linear(X_train, y_train, X_test, y_test):
    regressor = SVR(kernel='linear')
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)
# --------------------------------------------------
# 5. Non-linear SVR
# --------------------------------------------------
def svm_NL(X_train, y_train, X_test, y_test):
    regressor = SVR(kernel='rbf')
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)
# --------------------------------------------------
# 6. Decision Tree Regression
# --------------------------------------------------
def Decision(X_train, y_train, X_test, y_test):
    regressor = DecisionTreeRegressor(random_state=0)
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)
# --------------------------------------------------
# 7. Random Forest Regression
# --------------------------------------------------
def random(X_train, y_train, X_test, y_test):
    regressor = RandomForestRegressor(n_estimators=10,random_state=0)
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)

# 4. RFE Feature Selection

In [9]:
def rfeFeature(indep_X, dep_Y, n):
    rfelist = []
    feature_names = []
    rfemodellist = [LinearRegression(),SVR(kernel='linear'),DecisionTreeRegressor(random_state=0),RandomForestRegressor(
            n_estimators=10,random_state=0)]
    model_names = ['Linear','SVC','DecisionTree','Random']

    for i, model in enumerate(rfemodellist):
        print("\nRFE Feature Selection:", model_names[i])
        log_rfe = RFE(estimator=model,n_features_to_select=n)
        log_rfe.fit(indep_X, dep_Y)
        selected_features = indep_X.columns[log_rfe.support_].tolist()
        print("Selected Features:", selected_features)
        feature_names.append(model_names[i])
        log_rfe_feature = log_rfe.transform(indep_X)
        rfelist.append(log_rfe_feature)
    return rfelist, feature_names

# 5. Create the R2 comparison table

In [10]:
def rfe_regression(accl,accsvml,accsvmnl,accdes,accrf):
    rfedataframe = pd.DataFrame(index=['Linear','SVC','Random','DecisionTree'],
        columns=['Linear','SVMl','SVMnl','Decision','Random'])
    for number, index_name in enumerate(rfedataframe.index):
        rfedataframe.loc[index_name, 'Linear'] = accl[number]
        rfedataframe.loc[index_name, 'SVMl'] = accsvml[number]
        rfedataframe.loc[index_name, 'SVMnl'] = accsvmnl[number]
        rfedataframe.loc[index_name, 'Decision'] = accdes[number]
        rfedataframe.loc[index_name, 'Random'] = accrf[number]
    return rfedataframe


# 6. Load dataset

In [11]:
dataset1 = pd.read_csv("prep.csv", index_col=None)
df2 = dataset1.copy()
df2 = pd.get_dummies(df2, drop_first=True)
indep_X = df2.drop('classification_yes', axis=1)
dep_Y = df2['classification_yes']

# 7. Select three features using RFE

In [ ]:
rfelist, feature_names = rfeFeature(indep_X,dep_Y,3)
# Create empty lists for R2 scores
accl = []
accsvml = []
accsvmnl = []
accdes = []
accrf = []


RFE Feature Selection: Linear
Selected Features: ['sg_c', 'sg_d', 'sg_e']

RFE Feature Selection: SVC


# 8. Evaluate all regression algorithms

In [15]:
for selected_X in rfelist:
    X_train, X_test, y_train, y_test = split_scalar(selected_X,dep_Y)

    # Linear Regression
    accl.append(Linear(X_train, y_train, X_test, y_test))

    # Linear SVR
    accsvml.append(svm_linear(X_train, y_train, X_test, y_test))

    # Non-linear SVR
    accsvmnl.append(svm_NL(X_train, y_train, X_test, y_test))

    # Decision Tree
    accdes.append(Decision(X_train, y_train, X_test, y_test))

    # Random Forest
    accrf.append(random(X_train, y_train, X_test, y_test))
    
    # Display the final comparison table WITH INDEX
result = rfe_regression(accl,accsvml,accsvmnl,accdes,accrf)
result = result.astype(float)
print("\nR2 Score Comparison Table:\n")
print(result.round(6).to_string())



R2 Score Comparison Table:

                Linear      SVMl     SVMnl  Decision    Random
Linear        0.441961  0.262153  0.262162  0.441961  0.441816
SVC           0.441961  0.262153  0.262162  0.441961  0.441816
Random        0.441961  0.262153  0.262162  0.441961  0.441816
DecisionTree  0.664893  0.609652  0.883134  0.965961  0.916304


# 9. Find the best score

In [17]:
best_score = result.max().max()
best_position = np.where(result.values == best_score)
best_rfe_method = result.index[best_position[0][0]]
best_algorithm = result.columns[best_position[1][0]]
print("\nBest R2 Score:", round(best_score, 2))
print("Best RFE Method:", best_rfe_method)
print("Best Regression Algorithm:", best_algorithm)



Best R2 Score: 0.97
Best RFE Method: DecisionTree
Best Regression Algorithm: Decision
